### Model InterpretabilityAccuracy and explainability are separate axes, a model can be highly accurate and still be a black box. `decision-tree.ipynb`'s built-in feature importance (Gini-reduction, summed across splits) only works for tree models and only gives a GLOBAL view (which features matter overall). This notebook covers model-agnostic methods that work on any model, and that can explain a SINGLE prediction, not just the model as a whole, directly relevant to a fraud system where you need to say why THIS transaction got flagged, not just which features matter on average.

#### 0. Global vs. local, model-specific vs. model-agnosticTwo independent axes worth keeping separate:- Global: which features matter across the whole dataset (built-in tree importance, permutation importance averaged over many rows).- Local: why did the model predict THIS for THIS specific row (LIME, a single row's SHAP values).Model-specific methods (tree importance) only work for one model family but are cheap. Model-agnostic methods (permutation importance, LIME, SHAP) work on anything, logistic regression, XGBoost, a neural net, at the cost of being slower to compute.

#### 1. Permutation importance, worked by handModel-agnostic global importance: shuffle ONE feature's values across all rows (breaking its relationship with the target while keeping every other feature and the target intact), measure how much the model's performance drops. A feature the model actually relies on causes a big drop when scrambled, a feature it ignores causes almost none.Worked example: model scores 0.90 accuracy on a held-out fraud set. Shuffle `mentions_IRS` alone, rerun predictions with everything else unchanged, accuracy drops to 0.74. Shuffle `narrative_length` alone instead, accuracy barely moves, 0.895. `mentions_IRS` importance = 0.90 - 0.74 = 0.16, `narrative_length` importance = 0.90 - 0.895 = 0.005, the model is relying heavily on the first feature and barely at all on the second, a claim built-in Gini importance can't make for a non-tree model, and that even for a tree model measures split quality, not actual predictive reliance on held-out data.

In [ ]:
import numpy as npfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.inspection import permutation_importancefrom sklearn.datasets import make_classificationX, y = make_classification(n_samples=500, n_features=5, n_informative=3, random_state=42)model = RandomForestClassifier(random_state=42).fit(X, y)result = permutation_importance(model, X, y, n_repeats=10, random_state=42)for i, (mean, std) in enumerate(zip(result.importances_mean, result.importances_std)):    print(f"feature_{i}: importance={mean:.4f} +/- {std:.4f}")

#### 2. LIME (Local Interpretable Model-agnostic Explanations), worked by handExplains ONE prediction by fitting a simple, interpretable model (linear regression) LOCALLY around that single point, in a region small enough that even a wildly nonlinear black box looks approximately linear there. The surrogate's coefficients become the explanation.Worked example: black box model f(x) = x^2 (nonlinear, deliberately, to show LIME approximating a curve with a local line), explaining the point x0=3, f(x0)=9. LIME procedure:```1. Perturb: sample points near x0 -> x = [1, 2, 3, 4, 5], f(x) = [1, 4, 9, 16, 25]2. Weight: closer points to x0 count more, kernel weight = exp(-(x-x0)^2 / (2*sigma^2))   at sigma=1.5: weights = [0.038, 0.325, 1.0, 0.325, 0.038]  (point x=3 itself weighted highest)3. Fit: weighted linear regression y = a + b*x through these 5 points using those weights```The true local slope of f(x)=x^2 at x0=3 is the derivative f'(3) = 2*3 = 6 (see `calculus-differentiation.ipynb`). LIME's fitted slope b should land close to 6, the weighted regression is, by construction, approximating that local tangent line using only samples and predictions, no access to the model's internals or its actual derivative, this is the entire point, LIME never looks inside the black box, only at its outputs on perturbed inputs.

In [ ]:
import numpy as npx0, sigma = 3, 1.5x_samples = np.array([1, 2, 3, 4, 5])f = lambda x: x ** 2  # the "black box"y_samples = f(x_samples)weights = np.exp(-((x_samples - x0) ** 2) / (2 * sigma ** 2))print("perturbed x:", x_samples)print("black-box f(x):", y_samples)print("kernel weights (closer to x0 counts more):", weights.round(3))# weighted linear regression: y = a + b*x, via weighted least squaresX_design = np.vstack([np.ones_like(x_samples), x_samples]).TW = np.diag(weights)coeffs = np.linalg.inv(X_design.T @ W @ X_design) @ X_design.T @ W @ y_samplesa, b = coeffsprint(f"\nLIME local surrogate: y = {a:.3f} + {b:.3f}*x")print(f"true local slope at x0=3 (f'(x)=2x): {2*x0}")print(f"LIME's slope ({b:.3f}) approximates the true local derivative ({2*x0}), without ever seeing f's formula")

#### 3. SHAP (SHapley Additive exPlanations), worked by handBorrows Shapley values from cooperative game theory: treat each feature as a "player" contributing to the "payout" (the prediction, relative to a baseline), and fairly split that payout among the features based on each one's AVERAGE marginal contribution across every possible order the features could be "added" to the prediction.Formula, feature i's Shapley value: `phi_i = sum over subsets S not containing i of [|S|!(n-|S|-1)!/n!] * (f(S U {i}) - f(S))`, a weighted average of i's marginal contribution across every coalition it could join.Worked example, exact by hand: toy LINEAR model f(x1,x2) = 2*x1 + 3*x2 (chosen linear on purpose, so the true per-feature contribution is unambiguous and SHAP's answer can be checked against it), baseline x1=x2=0 (f(baseline)=0), explaining instance x1=1, x2=1 (f(x)=5).```f({})     = f(0,0) = 0f({1})    = f(1,0) = 2f({2})    = f(0,1) = 3f({1,2})  = f(1,1) = 5phi_1 = 0.5*[f({1})-f({})] + 0.5*[f({1,2})-f({2})] = 0.5*(2-0) + 0.5*(5-3) = 1 + 1 = 2phi_2 = 0.5*[f({2})-f({})] + 0.5*[f({1,2})-f({1})] = 0.5*(3-0) + 0.5*(5-2) = 1.5 + 1.5 = 3```Efficiency property check: phi_1 + phi_2 = 2 + 3 = 5 = f(x) - f(baseline), the Shapley values exactly account for the full prediction, no leftover, no double-counting. And since the model is linear, phi_1 = 2 = weight_1 * x1 and phi_2 = 3 = weight_2 * x2 exactly, SHAP recovers each feature's true contribution when the ground truth is known, the same guarantee LIME's local linear approximation only approaches asymptotically, not exactly.

In [ ]:
import itertoolsimport mathimport numpy as npdef f(x1, x2):    return 2 * x1 + 3 * x2baseline = {1: 0, 2: 0}instance = {1: 1, 2: 1}features = [1, 2]def value(subset):    x = {i: (instance[i] if i in subset else baseline[i]) for i in features}    return f(x[1], x[2])def shapley(feature, features):    others = [f for f in features if f != feature]    n = len(features)    total = 0.0    for r in range(len(others) + 1):        for subset in itertools.combinations(others, r):            subset = set(subset)            weight = (math.factorial(len(subset)) * math.factorial(n - len(subset) - 1)) / math.factorial(n)            marginal = value(subset | {feature}) - value(subset)            total += weight * marginal    return totalphi = {i: shapley(i, features) for i in features}print("Shapley values:", phi)print("sum of Shapley values:", sum(phi.values()), "| f(x) - f(baseline):", value({1,2}) - value(set()))

#### 4. TreeSHAP: making SHAP tractable for real modelsThe exact formula above sums over every possible feature subset, 2^n terms, computationally impossible past a handful of features (n=20 features -> over a million subsets, per row). TreeSHAP exploits tree structure (the same trees from `bagging.ipynb`/`boosting.ipynb`) to compute exact Shapley values in polynomial time instead, by tracking how each split routes different feature coalitions through the tree, rather than brute-forcing every subset. This is the practical reason SHAP became standard for tree ensembles (XGBoost/LightGBM/Random Forest) specifically, exact game-theoretic explanations at a computationally feasible cost, whereas non-tree models fall back to KernelSHAP (a sampling-based approximation, slower, no exactness guarantee).

In [ ]:
import shapimport numpy as npfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.datasets import make_classificationX, y = make_classification(n_samples=300, n_features=4, n_informative=3, random_state=42)model = RandomForestClassifier(random_state=42, n_estimators=50).fit(X, y)explainer = shap.TreeExplainer(model)shap_values = explainer.shap_values(X[:5])  # local explanation for the first 5 rows# local: this row's per-feature contribution to its own predictionprint("SHAP values for row 0 (class 1):", np.round(shap_values[1][0] if isinstance(shap_values, list) else shap_values[0], 3))# global: mean absolute SHAP value per feature, across all explained rows, same information# permutation importance gives, computed via the additive Shapley decomposition insteadmean_abs_shap = np.abs(shap_values[1] if isinstance(shap_values, list) else shap_values).mean(axis=0)print("global importance (mean |SHAP|):", np.round(mean_abs_shap, 3))

#### 5. SHAP vs. LIME: when each one breaks| | LIME | SHAP ||---|---|---|| Guarantee | none, a local linear approximation, quality depends on kernel width/sample count | game-theoretic guarantees (efficiency, symmetry, additivity), exact for TreeSHAP || Stability | can give different explanations on repeated runs, depends on random perturbation sampling | deterministic for TreeSHAP, exact same input gives same output || Speed | fast, one local regression per explanation | fast for trees (TreeSHAP), slow for arbitrary models (KernelSHAP samples many coalitions) || Model support | any model, always model-agnostic | exact+fast only for trees, model-agnostic fallback exists but is slow || Output | local surrogate coefficients | Shapley values, additive and exactly explain the prediction gap from baseline |Practical default: TreeSHAP for anything tree-based (XGBoost, LightGBM, Random Forest, the models this whole `fundamentals/` series leans on most), LIME or KernelSHAP when the model is something SHAP has no fast exact algorithm for and a quick, cheap local explanation is good enough.

#### 6. Tie-in: explaining a fraud flagThe concrete use case this whole notebook is really for: a fraud model flags a transaction, and the question isn't just "is this fraud" (the prediction itself, `eval-metrics.ipynb`'s territory) but "why did the model think so" for THIS specific case, a local explanation, not a global feature-importance ranking. TreeSHAP on an XGBoost fraud model gives exactly that: for one flagged transaction, `mentions_IRS` contributed +0.31 to the fraud-probability logit, `narrative_length` contributed +0.02, `time_of_day` contributed -0.05, and the baseline (average model output across the training set) plus those contributions sums exactly to this transaction's predicted score, additive and auditable, the property a compliance review or a fraud analyst actually needs, not just an accuracy number.